# 04 — LSTM Fine-Tuning

**Goal:** Train and compare three LSTM configurations (Baseline, Lightweight, Regularized) with **class weighting** and **Early Stopping**. Save the best model + validation predictions for Notebook 05.


## 4.1 Imports & load features


In [ ]:
import os, pickle
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.utils.class_weight import compute_class_weight

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # quiet TF
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers, callbacks

tf.get_logger().setLevel('ERROR')
tf.random.set_seed(42)
np.random.seed(42)

ROOT = Path.cwd()
for _ in range(6):
    if (ROOT / 'Data' / 'cryptonews.csv').exists() or (ROOT / 'notebooks').exists():
        break
    if ROOT == ROOT.parent:
        break
    ROOT = ROOT.parent
INTERIM = ROOT / 'notebooks' / 'interim'

with (INTERIM / 'features_for_lstm.pkl').open('rb') as f:
    bundle = pickle.load(f)

train_x, train_y = bundle['train_x'], bundle['train_y']
val_x,   val_y   = bundle['val_x'],   bundle['val_y']
test_x,  test_y  = bundle['test_x'],  bundle['test_y']
n_features = train_x.shape[-1]

print(f'train: {train_x.shape}  val: {val_x.shape}  test: {test_x.shape}')
print(f'features: {n_features}')
print(f'class balance (train): up={train_y.mean():.3f}  down={1-train_y.mean():.3f}')

## 4.2 Class weights
5-day directional targets are imbalanced (bull markets dominate). We pass balanced class weights to `model.fit` so the LSTM doesn't collapse to majority-class predictions.


In [ ]:
classes = np.unique(train_y)
weights = compute_class_weight('balanced', classes=classes, y=train_y)
class_weight = {int(c): float(w) for c, w in zip(classes, weights)}
print('Class weights:', class_weight)

## 4.3 Three LSTM configurations
A lightweight hyperparameter search: three configs covering the bias-variance tradeoff.

| Config       | Layers    | Units   | Dropout | L2 reg   | Bidirectional |
|--------------|-----------|---------|---------|----------|---------------|
| Baseline     | 1 LSTM    | 64      | 0.0     | 0.0      | No            |
| Lightweight  | 1 LSTM    | 32      | 0.1     | 0.0      | No            |
| Regularized  | 2 LSTM    | 64 → 32 | 0.3     | 1e-4     | No            |
| Bi-LSTM      | 1 BiLSTM  | 64      | 0.2     | 1e-5     | Yes           |


In [ ]:
def build_model(config: str, n_features: int) -> tf.keras.Model:
    """Build one of the four LSTM configurations."""
    inp = layers.Input(shape=(1, n_features), name='features')
    x = inp
    if config == 'baseline':
        x = layers.LSTM(64, return_sequences=False)(x)
    elif config == 'lightweight':
        x = layers.LSTM(32, return_sequences=False, dropout=0.1)(x)
    elif config == 'regularized':
        x = layers.LSTM(64, return_sequences=True,
                        kernel_regularizer=regularizers.l2(1e-4),
                        recurrent_dropout=0.2)(x)
        x = layers.Dropout(0.3)(x)
        x = layers.LSTM(32, return_sequences=False,
                        kernel_regularizer=regularizers.l2(1e-4))(x)
        x = layers.Dropout(0.3)(x)
    elif config == 'bilstm':
        x = layers.Bidirectional(layers.LSTM(64, return_sequences=False, dropout=0.2,
                                              kernel_regularizer=regularizers.l2(1e-5)))(x)
    else:
        raise ValueError(config)
    x = layers.Dense(16, activation='relu')(x)
    out = layers.Dense(1, activation='sigmoid', name='prob_up')(x)
    m = models.Model(inp, out, name=config)
    m.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss='binary_crossentropy',
              metrics=['accuracy', tf.keras.metrics.AUC(name='auc')])
    return m

# Smoke test — build all four and print param counts
for cfg in ['baseline', 'lightweight', 'regularized', 'bilstm']:
    m = build_model(cfg, n_features)
    print(f'{cfg:12s}  params={m.count_params():>7,}')
    m._backend = None  # release

## 4.4 Train all four configs with Early Stopping
Each config trains for up to 50 epochs with Early Stopping (patience=7) on validation AUC. The best epoch (by val_auc) is restored.


In [ ]:
CONFIGS = ['baseline', 'lightweight', 'regularized', 'bilstm']
EPOCHS = 50
BATCH = 32

histories = {}
models_dir = INTERIM / 'models'
models_dir.mkdir(exist_ok=True)

for cfg in CONFIGS:
    print(f'\n===== Training {cfg} =====')
    tf.keras.backend.clear_session()
    tf.random.set_seed(42)
    model = build_model(cfg, n_features)
    es = callbacks.EarlyStopping(monitor='val_auc', mode='max', patience=7, restore_best_weights=True)
    rlrop = callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-5)
    hist = model.fit(
        train_x, train_y,
        validation_data=(val_x, val_y),
        epochs=EPOCHS, batch_size=BATCH,
        class_weight=class_weight,
        callbacks=[es, rlrop],
        verbose=2,
    )
    histories[cfg] = hist.history
    model.save(models_dir / f'{cfg}.ker')
    print(f'  best val_auc = {max(hist.history["val_auc"]):.4f}  best val_acc = {max(hist.history["val_accuracy"]):.4f}')

## 4.5 Compare configs on validation set


In [ ]:
rows = []
for cfg in CONFIGS:
    h = histories[cfg]
    rows.append({
        'config': cfg,
        'val_auc': max(h['val_auc']),
        'val_acc': max(h['val_accuracy']),
        'val_loss': min(h['val_loss']),
        'epochs_run': len(h['val_loss']),
        'params': build_model(cfg, n_features).count_params(),
    })
    tf.keras.backend.clear_session()
summary = pd.DataFrame(rows).sort_values('val_auc', ascending=False)
summary

## 4.6 Persist predictions on val + test for Notebook 05


In [ ]:
preds = {}
for cfg in CONFIGS:
    tf.keras.backend.clear_session()
    m = tf.keras.models.load_model(models_dir / f'{cfg}.ker')
    preds[cfg] = {
        'val_prob':  m.predict(val_x,  verbose=0).ravel(),
        'test_prob': m.predict(test_x, verbose=0).ravel(),
    }
    print(f'{cfg:12s}  test mean prob = {preds[cfg]["test_prob"].mean():.3f}  test std = {preds[cfg]["test_prob"].std():.3f}')

with (INTERIM / 'lstm_predictions.pkl').open('wb') as f:
    pickle.dump({'preds': preds, 'val_y': val_y, 'test_y': test_y,
                 'val_dates': bundle['val_dates'], 'test_dates': bundle['test_dates'],
                 'test_close': bundle['test_close'],
                 'test_forward_ret_5d': bundle['test_forward_ret_5d']}, f)
print('Wrote predictions to', INTERIM / 'lstm_predictions.pkl')

## 4.7 Summary
- Trained four LSTM configs (Baseline, Lightweight, Regularized, Bi-LSTM).
- Used `class_weight='balanced'` to address target imbalance.
- Early Stopping (patience=7) + ReduceLROnPlateau for stable convergence.
- Best config selected by validation AUC; all four predictions persisted for Notebook 05.
